In [3]:
from dotenv import load_dotenv
import os
load_dotenv()
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_URL = os.getenv("QDRANT_URL")
DATABASE_NAME = os.getenv("DATABASE_NAME")

In [4]:
from qdrant_client import QdrantClient
import qdrant_client
def get_qdrant_client():
    return QdrantClient(
        url=QDRANT_URL,
        api_key=QDRANT_API_KEY
    )
    
client = get_qdrant_client()

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

text = "hello world"

vector = embedding_model.embed_query(text)

vector_size = len(vector)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10848.88it/s]


In [9]:
#create collection if it doesn't exixts

from qdrant_client.models import VectorParams,Distance

def create_collection_if_not_exists():
    collections = client.get_collections().collections

    collections_name = [collection.name for collection in collections]

    if DATABASE_NAME not in collections_name:
        client.create_collection(
            collection_name=DATABASE_NAME,
            vectors_config=VectorParams(
                size=vector_size,
                distance=Distance.COSINE
            )
        )

        print("✅ Collection Created")

       

        print("✅ Payload Indexes Created")
    else:
        print("ℹ️ Collection Already Exists")

create_collection_if_not_exists()


✅ Collection Created
✅ Payload Indexes Created


In [10]:
from qdrant_client.models import PointStruct
import uuid

point = PointStruct(
    id = str(uuid.uuid4()),
    vector=vector,
    payload={
        "page_content":text,
        "department":"Insurance",
        "source":"Insurance.pdf",
        "page":1
    }
)

In [11]:
client.upsert(
    collection_name=DATABASE_NAME,
      wait=True,
      points=[point]
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)